# Shoes vs Sandals: self-built CNN and transfer learning

**Task:** classify a photo as a closed **shoe** or as a **sandal** (sandals, chappals, Peshawari chappal, slippers and flip-flops are all counted as "sandals").

**Data:** photos and video frames collected by the student from shops, offices and contacts (primary dataset).

**Models:** (a) a small CNN built and trained from scratch, (b) MobileNetV2 pretrained on ImageNet, fine-tuned (transfer learning).

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. If the `dataset` folder is not in the session, the first cell asks you to upload your dataset zip (`dataset_public.zip`).

**Honest evaluation:** whole videos are kept in train OR test, so test frames never come from a video the model trained on. `test_single` holds single-item photos of pairs that are not in train.

In [ ]:
import os, zipfile
if not os.path.isdir("dataset"):
    from google.colab import files
    print("Upload your dataset zip (dataset_public.zip) when the button appears")
    uploaded = files.upload()
    with zipfile.ZipFile(list(uploaded)[0]) as z:
        z.extractall(".")
for split in ["train", "test", "test_single"]:
    for c in ["shoes", "sandals"]:
        print(split, c, len(os.listdir(f"dataset/{split}/{c}")))

In [ ]:
import glob, re, json, random, time, subprocess
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG = 224
BATCH = 32
CLASSES = ["sandals", "shoes"]      # label 0 = sandals, label 1 = shoes
SINGLE_WEIGHT = 5.0                 # single photos count 5x a video frame in training
EPOCHS_CNN = int(os.environ.get("EPOCHS_CNN", 40))
EPOCHS_HEAD = int(os.environ.get("EPOCHS_HEAD", 6))
EPOCHS_FT = int(os.environ.get("EPOCHS_FT", 6))
PRETRAINED = os.environ.get("PRETRAINED", "imagenet")
if PRETRAINED == "none":
    PRETRAINED = None
AUTOTUNE = tf.data.AUTOTUNE
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Data preparation

Steps: fix sideways phone photos (EXIF rotation), list files per split, resize every image to 224x224, scale pixels to the range -1..1, and augment training images only.

In [ ]:
# Phone photos can store "rotate me" in EXIF data. TensorFlow ignores it, so fix the pixels once.
from PIL import Image, ImageOps
fixed = 0
for p in glob.glob("dataset/**/*", recursive=True):
    name = os.path.basename(p)
    if p.lower().endswith((".jpg", ".jpeg")) and not re.search(r"_take\d+_\d+", name):
        im = Image.open(p)
        if im.getexif().get(274, 1) != 1:
            ImageOps.exif_transpose(im).convert("RGB").save(p, quality=95)
            fixed += 1
print("photos with sideways orientation fixed:", fixed)

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png")

def is_frame(path):
    # video frames are named like shoe_take1_0003.jpg, single photos have other names
    return re.search(r"_take\d+_\d+", os.path.basename(path)) is not None

def list_split(split):
    paths, labels, kinds = [], [], []
    for ci, c in enumerate(CLASSES):
        for p in sorted(glob.glob(f"dataset/{split}/{c}/*")):
            if p.lower().endswith(IMG_EXT):
                paths.append(p); labels.append(ci); kinds.append(is_frame(p))
    return paths, np.array(labels, np.float32), np.array(kinds)

def load_image(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG, IMG))      # plain resize, no crop (same as the web app)
    img.set_shape((IMG, IMG, 3))
    return img                                   # float32, values 0..255

def normalize(x):
    return x / 127.5 - 1.0                       # -> range -1..1 (same as the web app)

# Augmentation (training images only). Zoom is mostly "zoom in" so a rack of shoes
# starts to look like a few shoes, closer to the single photos users will upload.
AUG = {
    # "strong" is used for transfer learning (a pretrained network copes well with it)
    "strong": keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(height_factor=(-0.45, 0.10)),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.2, value_range=(0, 255)),
    ], name="augment_strong"),
    # "mild" is used for the CNN built from scratch, which has to learn from zero
    "mild": keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.04),
        layers.RandomZoom(height_factor=(-0.25, 0.05)),
        layers.RandomContrast(0.1),
        layers.RandomBrightness(0.1, value_range=(0, 255)),
    ], name="augment_mild"),
}

def train_dataset(aug="strong"):
    p, l, kinds = list_split("train")
    w = np.where(kinds, 1.0, SINGLE_WEIGHT).astype(np.float32)
    for ci in (0, 1):                            # balance the two classes
        w[l == ci] *= len(l) / (2.0 * np.sum(l == ci))
    ds = tf.data.Dataset.from_tensor_slices((p, l, w))
    # decode and resize each file once, keep the small uint8 copy in memory (much faster epochs)
    ds = ds.map(lambda path, y, wt: (tf.cast(tf.round(load_image(path)), tf.uint8), y, wt),
                num_parallel_calls=AUTOTUNE).cache()
    ds = ds.shuffle(len(p), seed=SEED).batch(BATCH)
    aug_layer = AUG[aug]
    ds = ds.map(lambda x, y, wt: (normalize(aug_layer(tf.cast(x, tf.float32), training=True)), y, wt),
                num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

def eval_dataset(split):
    p, l, _ = list_split(split)
    ds = tf.data.Dataset.from_tensor_slices((p, l))
    return ds.map(lambda path, y: (normalize(load_image(path)), y),
                  num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

for split in ["train", "test", "test_single"]:
    p, l, kinds = list_split(split)
    print(f"{split:12s} total {len(p):5d} | shoes {int(l.sum()):5d} | sandals {int(len(l) - l.sum()):5d} | "
          f"single photos {int((~kinds).sum()):4d} | video frames {int(kinds.sum()):5d}")

In [ ]:
# Look at a few training images and their augmented versions
p, l, _ = list_split("train")
idx = random.sample(range(len(p)), 8)
fig, axes = plt.subplots(3, 8, figsize=(20, 7.5))
for j, i in enumerate(idx):
    img = load_image(p[i])
    axes[0, j].imshow(img.numpy().astype("uint8")); axes[0, j].set_title(CLASSES[int(l[i])]); axes[0, j].axis("off")
    for row, key in ((1, "mild"), (2, "strong")):
        aug_img = AUG[key](img[None], training=True)[0].numpy()   # same batched input shape as in training
        axes[row, j].imshow(np.clip(aug_img, 0, 255).astype("uint8")); axes[row, j].axis("off")
plt.suptitle("Row 1: original. Row 2: mild augmentation (CNN). Row 3: strong augmentation (transfer learning)")
plt.savefig("results/augmentation_examples.png", dpi=110, bbox_inches="tight")
plt.show()

## 2. The two models

**Self-built CNN:** four blocks of Conv, BatchNorm, ReLU, MaxPool (32, 64, 128, 128 filters), then global average pooling, dropout and one sigmoid output. Weights start random.

**Transfer learning:** MobileNetV2 with ImageNet weights as the feature extractor, plus a new dropout and sigmoid output. Stage 1 trains only the new head. Stage 2 unfreezes the last 30 layers of MobileNetV2 and trains them gently (learning rate 1e-5).

Output of both models: a number between 0 and 1 = probability that the photo is a **shoe**.

In [ ]:
def build_cnn():
    inp = keras.Input((IMG, IMG, 3))
    x = inp
    for f in [32, 64, 128, 128]:
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization(momentum=0.9)(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inp, out, name="self_cnn")

def build_transfer():
    base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights=PRETRAINED)
    base.trainable = False
    inp = keras.Input((IMG, IMG, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inp, out, name="transfer_mobilenetv2"), base

def compile_model(m, lr):
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])

cnn = build_cnn()
cnn.summary()

## 3. Train the self-built CNN

The `test` folder (whole videos the model never trains on) is shown during training only to draw the curves. It is **not** used to pick epochs or settings, because the number of epochs is fixed in advance.

The CNN learns from zero, so it gets milder augmentation than the transfer model, 40 epochs, and a learning rate that glides from 1e-3 down to 0 (cosine decay). Decoded images are cached in memory, which makes every epoch after the first much faster.

In [ ]:
train_ds_cnn = train_dataset("mild")
val_ds = eval_dataset("test")
steps_per_epoch = len(train_ds_cnn)
# learning rate starts at 1e-3 and glides down to 0, so the last epochs are calm and stable
lr_schedule = keras.optimizers.schedules.CosineDecay(1e-3, decay_steps=EPOCHS_CNN * steps_per_epoch)
compile_model(cnn, lr_schedule)
t0 = time.time()
hist_cnn = cnn.fit(train_ds_cnn, validation_data=val_ds, epochs=EPOCHS_CNN, verbose=2)
print(f"CNN training time: {(time.time() - t0) / 60:.1f} min")

## 4. Train the transfer-learning model

In [ ]:
train_ds_tl = train_dataset("strong")
tl, base = build_transfer()
compile_model(tl, 1e-3)
t0 = time.time()
print("Stage 1: train the new head only")
h1 = tl.fit(train_ds_tl, validation_data=val_ds, epochs=EPOCHS_HEAD, verbose=2)

print("Stage 2: fine-tune the last 30 layers of MobileNetV2")
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
compile_model(tl, 1e-5)
h2 = tl.fit(train_ds_tl, validation_data=val_ds, epochs=EPOCHS_FT, verbose=2)
print(f"Transfer training time: {(time.time() - t0) / 60:.1f} min")
hist_tl = {k: h1.history[k] + h2.history[k] for k in h1.history}

## 5. Results

Two test sets: `test` (frames from videos not used for training) and `test_single` (single photos of pairs not used for training). `test_single` is the closest to what a user does on the website.

In [ ]:
def evaluate(model, split):
    _, l, _ = list_split(split)
    probs = model.predict(eval_dataset(split), verbose=0).ravel()
    preds = (probs >= 0.5).astype(int)
    return {"n": int(len(l)),
            "accuracy": float((preds == l).mean()),
            "confusion_matrix": confusion_matrix(l, preds, labels=[0, 1]).tolist(),
            "report": classification_report(l, preds, labels=[0, 1], target_names=CLASSES,
                                            output_dict=True, zero_division=0)}

results = {}
for name, m in [("self_cnn", cnn), ("transfer_mobilenetv2", tl)]:
    results[name] = {split: evaluate(m, split) for split in ["test", "test_single"]}

print(f"{'model':24s} {'test (video frames)':>22s} {'test_single (photos)':>24s}")
for name, r in results.items():
    print(f"{name:24s} {r['test']['accuracy'] * 100:15.1f}% (n={r['test']['n']}) "
          f"{r['test_single']['accuracy'] * 100:15.1f}% (n={r['test_single']['n']})")
print("\nWith so few test images, one wrong image moves accuracy by several points. Read the numbers as rough.")

json.dump(results, open("results/results.json", "w"), indent=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (title, h) in zip(axes, [("Self-built CNN", hist_cnn.history), ("Transfer learning", hist_tl)]):
    ax.plot(h["accuracy"], label="train accuracy")
    ax.plot(h["val_accuracy"], label="test (video frames) accuracy")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylim(0, 1.02); ax.legend()
plt.savefig("results/training_curves.png", dpi=110, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for row, name in enumerate(["self_cnn", "transfer_mobilenetv2"]):
    for col, split in enumerate(["test", "test_single"]):
        cm = np.array(results[name][split]["confusion_matrix"])
        ax = axes[row, col]
        ax.imshow(cm, cmap="Blues")
        ax.set_xticks([0, 1]); ax.set_xticklabels(CLASSES); ax.set_yticks([0, 1]); ax.set_yticklabels(CLASSES)
        ax.set_title(f"{name}\n{split}"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.tight_layout()
plt.savefig("results/confusion_matrices.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# Show the single photos each model got wrong (useful for the report)
p, l, _ = list_split("test_single")
probs = {n: m.predict(eval_dataset("test_single"), verbose=0).ravel() for n, m in [("self_cnn", cnn), ("transfer", tl)]}
for n, pr in probs.items():
    wrong = [i for i in range(len(l)) if int(pr[i] >= 0.5) != int(l[i])]
    print(f"{n}: {len(wrong)} wrong out of {len(l)}")
    for i in wrong[:10]:
        print("   ", os.path.basename(p[i]), "true:", CLASSES[int(l[i])], "| P(shoe) =", round(float(pr[i]), 2))

## 6. Save the weights and the model files

Saved: the full models (`.keras`) and the weights only (`.weights.h5`), plus `classes.json` that says how to feed an image to the models. These are the files the GitHub repository needs.

In [ ]:
cnn.save("models/self_cnn.keras")
cnn.save_weights("models/self_cnn.weights.h5")
tl.save("models/transfer_mobilenetv2.keras")
tl.save_weights("models/transfer_mobilenetv2.weights.h5")
json.dump({
    "classes": {"0": "sandals", "1": "shoes"},
    "note": "sandals includes chappals, Peshawari chappal, slippers and flip-flops",
    "input_size": IMG,
    "preprocessing": "resize to 224x224 (no crop), then x / 127.5 - 1",
    "output": "one sigmoid number = probability the photo is a shoe (>= 0.5 means shoe)",
}, open("models/classes.json", "w"), indent=2)

# sanity check: reload from disk and confirm the same predictions
x, _ = next(iter(eval_dataset("test_single")))
for name, m in [("self_cnn", cnn), ("transfer_mobilenetv2", tl)]:
    reloaded = keras.models.load_model(f"models/{name}.keras")
    diff = float(np.abs(m.predict(x, verbose=0) - reloaded.predict(x, verbose=0)).max())
    print(name, "max difference after reload:", diff)
print(os.listdir("models"))

## 7. Export for the web app (TensorFlow.js)

The website runs the models inside the visitor's browser, so Vercel needs no Python server.

In [ ]:
subprocess.run(["pip", "install", "-q", "tensorflowjs"], check=True)

# tensorflowjs_converter breaks on newer NumPy / TensorFlow because of two optional imports.
# This tiny wrapper patches them first, then runs the normal converter.
CONVERT_WRAPPER = '\n'.join([
    "import sys, types",
    "import numpy as np",
    "for alias, real in (('object', object), ('bool', bool), ('int', int), ('float', float)):",
    "    if alias not in np.__dict__:",
    "        setattr(np, alias, real)",
    "sys.modules.setdefault('tensorflow_hub', types.ModuleType('tensorflow_hub'))",
    "from tensorflowjs.converters.converter import pip_main",
    "sys.argv = ['tensorflowjs_converter'] + sys.argv[1:]",
    "pip_main()",
])
open("tfjs_convert.py", "w").write(CONVERT_WRAPPER)

for name, m in [("self_cnn", cnn), ("transfer_mobilenetv2", tl)]:
    m.export(f"saved_models/{name}")
    subprocess.run(["python3", "tfjs_convert.py", "--input_format=tf_saved_model",
                    "--output_format=tfjs_graph_model",
                    f"saved_models/{name}", f"web_models/{name}"], check=True)
    print(name, "->", sorted(os.listdir(f"web_models/{name}")))

In [ ]:
# Zip everything the next steps need and download it
subprocess.run("zip -rq outputs.zip models results web_models", shell=True, check=True)
print("outputs.zip size (MB):", round(os.path.getsize("outputs.zip") / 1e6, 1))
try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    pass